In [ ]:
# Make sure the notebook can import the modules:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

In [ ]:
import re
import pandas as pd
import pdfplumber


In [ ]:
from plausibility import run_checks
from aggregate import aggregate
from query import select, distinct

### Load data: 

In [ ]:
ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "data" / "all_consolidated.csv")
#df = pd.read_csv(ROOT / "data" / "all_consolidated_corr.csv")

In [ ]:
pub_info_table = pd.read_csv(ROOT / "data" / "pub_info_table.csv")

### Run plausibility check function from Claude:

In [ ]:
flags = run_checks(df)        # prints a report AND returns the flags table

#### Cave! run_checks(df)  only compares values to reported maxima and minima (does a value exceed an extremum?). This does nothing in cases where no maximal or minima were reported. 

### Develop more plausibility checks: 

#### CTQ items have a range (1-5) and item scores sum up to subscale scores, therefore CTQ records with values < 5 can only be item scores:  

In [ ]:
# Select a CTQ publication, sample type data type, and apply condition value < 5: 
publication_bools = (df.publication == 'Garcia-Fernandez et al. 2024')
sample_type_bools = (df.sample_type == 'patients')
subsample_bools = (df.subsample == 'whole_sample')
data_type_bools = (df.data_type == 'mean')
value_bools = (df.value < 5)
#df[publication_bools & sample_type_bools & subsample_bools & data_type_bools & value_bools]

In [ ]:
# Expected output: only record type item
set(df[publication_bools & sample_type_bools & subsample_bools & data_type_bools & value_bools].record_type)


In [ ]:
# Expected output: Empty dataframe: 
record_type_bools = (df.record_type != 'item')
df[publication_bools & sample_type_bools & subsample_bools & data_type_bools & value_bools & record_type_bools]

#### Healthy controls do not have schizophrenia nor any mental illness. Therefore the columns sample_with_mental_illness and sample_with_schizophrenia must sum to zero:

In [ ]:
# Expected value: 0
print('Expected output: 0')
print('Output:')
sum(df[df.sample_type=='healthy_controls'].sample_with_mental_illness)

In [ ]:
# Expected value: 0
print('Expected output: 0')
print('Output:')
sum(df[df.sample_type=='healthy_controls'].sample_with_schizophrenia)

### The columns sample_with_mental_illness and sample_with_schizophrenia can only have values 0 or 1:

In [ ]:
print(set(df.sample_with_mental_illness))
print(set(df.sample_with_schizophrenia))

### By definition in our schema the same subsample must have the same sample_size:

In [ ]:
# Data grouped by publication, sample_type, and subsample can only have one sample size.
# Expected output: 1
type(df.groupby(['publication', 'subsample']))
set(df.groupby(['publication', 'sample_type', 'subsample'])['sample_size'].nunique())

#### Publications that contain samples with mental illness must be flagged as such in the pub_info_table. The two sets below must be equal:

In [ ]:
# Make a primary key for row comparison:
df['fact_id'] = df.index

In [ ]:
# Filter both the fact table (df) and pub_info_table for sample_with_mental_illness==1:
df_mental_ill = df[df.sample_with_mental_illness==1]
pub_info_mental_ill = pub_info_table[pub_info_table.sample_with_mental_illness==1]

In [ ]:
# Get the publication set from fact table filtered for mental illness: 
pub_set_from_df = set(df_mental_ill.publication)

In [ ]:
# Get the publication set from the pub_info_table filtered for mental illness: 
pub_set_from_pub_table = set(pub_info_mental_ill.publication)

In [ ]:
# The two sets must be equal:
pub_set_from_df == pub_set_from_pub_table

### Cave! An inner join between pub_info_table and df (fact table) does NOT accurately filter for mental illness and schizophrenia. 

pub_info_table contains information at the publication level and not at the sample_type or subsample level! Mental illness and schizophrenia are present in sample_type patients but not in sample_type healthy_control. The fact that these columns are present in pub_info_table and in the fact table is probably a design flaw. The columns that differenciate between healthy_controls and patients should be named differently from the columns that flag the occurrence of mental illness/schizophrenia at a publication level. 

In [ ]:
# Filtering for patients in the fact table includes records referring to medical conditions
# other than mental illness. Merging (inner join) the fact table filtered for patients with the pub_info_table
# filtered for mental illness yields the fact table containing only records referring to mentally ill
# sample_types/subsamples. I.e. it must yield the same records as filtering directly for mental illness
# in the fact table:

df_patients = df[df.sample_type=='patients']
merged_df = pd.merge(pub_info_mental_ill, df_patients, on='pub_id', how='inner')
merged_df
print(merged_df.shape)
print(df_mental_ill.shape)
print(list(merged_df.fact_id) == list(df_mental_ill.fact_id))

In [ ]:
# The publication columns coming from the two different tables must be equal in the merged table:
print(merged_df.shape)
print(sum(merged_df.publication_x == merged_df.publication_y))


In [ ]:
# The two sets must be equal:
pub_set_from_merged = set(merged_df.publication_x)
pub_set_from_merged == pub_set_from_pub_table

### Inspect the data using the query module:

#### Module containing functions that mimic SQL functionalities:

In [ ]:
# see what's available to filter on
print(distinct(df, "scale"))

In [ ]:
distinct(df, "sample_type")

In [ ]:
# SQL-like: SELECT * WHERE scale='DES-T' AND sample='patients' AND data_type='mean'
#select(df, scale="DES-T", sample="patients", data_type="mean")
data_slice = select(df, scale="DERS", sample_type="healthy_controls", data_type="mean")
print('Dimensions of selected data slice:')
print(data_slice.shape)
data_slice.head()

In [ ]:
# Get unique values: 
distinct(df, "data_type")

In [ ]:
publication_bools = (df.publication == 'Garcia-Fernandez et al. 2024')
df[publication_bools].shape

In [ ]:
publication_bools = (df.publication == 'Giromini et al. 2012')
subsample_bools = (df.subsample== 'female')
df[publication_bools & subsample_bools].shape

### Visual value spot checks against the original papers: 

#### Check table 3 from CTQ_SF_GarciaFernandez_et_al_2024:

In [ ]:
img_path = ROOT / 'data' / 'screenshots_spot_checks' / 'CTQ_SF_GarciaFernandez_et_al_2024_table_3.png'
from IPython.display import Image
Image(img_path, width=500)

In [ ]:
print("Expected value from pdf:")
print("Subscale: Emotional abuse; mean value: 15.26")

In [ ]:
means = df[df.data_type== "mean"]
means_garcia_etal = means[means.publication== "Garcia-Fernandez et al. 2024"]
#print(means_garcia_etal[means_garcia_etal.scale_old == "CTQ-SF_emotional_abuse"].value)
subscale_bools = (means_garcia_etal.subscale== "EA")
record_type_bools = (means_garcia_etal.record_type=='subscale')
print(means_garcia_etal[subscale_bools & record_type_bools].value)

In [ ]:
print("Expected value from pdf:")
print("Subscale: Emotional neglect; mean value: 13.49")

In [ ]:
means = df[df.data_type== "mean"]
means_garcia_etal = means[means.publication== "Garcia-Fernandez et al. 2024"]
#print(means_garcia_etal[means_garcia_etal.scale_old == "CTQ-SF_emotional_neglect"].value)
subscale_bools = (means_garcia_etal.subscale== "EN")
record_type_bools = (means_garcia_etal.record_type=='subscale')
print(means_garcia_etal[subscale_bools & record_type_bools].value)

#### Check table 1 from CTQ_SF_GarciaFernandez_et_al_2024:

In [ ]:
img_path = ROOT / 'data' / 'screenshots_spot_checks' / 'CTQ_SF_GarciaFernandez_et_al_2024_table_1.png'
from IPython.display import Image
Image(img_path, width=500)

In [ ]:
print("Expected value from pdf:")
print("Subscale: Sexual abuse; item 24; mean value 1.865")
print("Subscale: Sexual abuse; item 24; standard deviation 1.458")

In [ ]:
means = df[df.data_type== "mean"]
means_garcia_etal = means[means.publication== "Garcia-Fernandez et al. 2024"]
#print(means_garcia_etal[means_garcia_etal.scale_old == "CTQ-SF_sexual_abuse_item_24"].value)
subscale_bools = (means_garcia_etal.subscale== "SA")
record_type_bools = (means_garcia_etal.record_type=='item')
print(means_garcia_etal[subscale_bools & record_type_bools][['scale', 
                                                            'subscale', 
                                                            'item_name',
                                                            'value', 
                                                            'data_type']])
print(means_garcia_etal[means_garcia_etal.item_name == "CTQ-SF_sexual_abuse_item_24"].value)

In [ ]:
standard_deviations = df[df.data_type== "sd"]
sds_garcia_etal = standard_deviations[standard_deviations.publication== "Garcia-Fernandez et al. 2024"]
#sds_garcia_etal[sds_garcia_etal.scale_old == "CTQ-SF_sexual_abuse_item_24"].value
sds_garcia_etal[sds_garcia_etal.item_name == "CTQ-SF_sexual_abuse_item_24"].value


#### Check results from PSYRATS_Favrod_et_al_2012:

In [ ]:
img_path = ROOT / 'data' / 'screenshots_spot_checks' / 'PSYRATS_Favrod_et_al_results.png'
from IPython.display import Image
Image(img_path, width=500)

In [ ]:
print("Expected value from pdf:")
print("Subscale: PSYRATS AS (auditory hallucination scale); mean value 26.5")
print("Subscale: PSYRATS AS (auditory hallucination scale); standard deviation 7.6")

In [ ]:
means = df[df.data_type== "mean"]
means_favrod_etal = means[means.publication== "Favrod et al. 2012"]
#print(means_favrod_etal[means_favrod_etal.scale_old == 'PSYRATS-AS_total'].value)
subscale_bools = (means_favrod_etal.subscale== "AHS")
record_type_bools = (means_favrod_etal.record_type=='subscale')
print(means_favrod_etal[subscale_bools & record_type_bools].value)

In [ ]:
means_favrod_etal.scale

In [ ]:
standard_deviations = df[df.data_type== "sd"]
sds_favrod_etal = standard_deviations[standard_deviations.publication== "Favrod et al. 2012"]

subscale_bools = (sds_favrod_etal.subscale== "AHS")
record_type_bools = (sds_favrod_etal.record_type=='subscale')
print(sds_favrod_etal[subscale_bools & record_type_bools].value)

#### Check results from DES-T_Modestin_et_al_2004:

In [ ]:
img_path = ROOT / 'data' / 'screenshots_spot_checks' / 'DES-T_Modestin_et_al_2004_table_1.png'
from IPython.display import Image
Image(img_path, width=700)

In [ ]:
print("Expected value from pdf:")
print("Scale: DES-T; sample: Nonpatients; mean value: 5.0; standard deviation: 9.3")

In [ ]:
means = df[df.data_type== "mean"]
means_modestin_etal = means[means.publication== "Modestin & Erni 2004"]
controls = means_modestin_etal[means_modestin_etal['sample_type']== 'healthy_controls']
controls_all = controls[controls['subsample']=='whole_sample']
controls_all.value

In [ ]:
standard_deviations = df[df.data_type== "sd"]
sds_modestin_etal = standard_deviations[standard_deviations.publication== "Modestin & Erni 2004"]
controls = sds_modestin_etal[sds_modestin_etal['sample_type']== 'healthy_controls']
controls_all = controls[controls['subsample']=='whole_sample']
controls_all.value

#### Check results from CTQ_SF_Xiang_etal_2021_long.csv

In [ ]:
img_path = ROOT / 'data' / 'screenshots_spot_checks' / 'CTQ_SF_Xiang_table_1.png'
from IPython.display import Image
Image(img_path, width=900)

In [ ]:
print('Expected value from pdf:')
print('Scale: CTQ-SF, subscale: emotional abuse (EA), subsample: baseline')
print('mean: 9.23; standard deviation: 3.49')

In [ ]:
pub_bools = df.publication=='Xiang et al. 2021'
subscale_bools = df.subscale=='EA'
subsample_bools = df.subsample=='baseline'
data_slice = df[pub_bools & subscale_bools & subsample_bools]
print('Mean value:')
print(data_slice[data_slice.data_type=='mean'].value)
print('Standard deviation:')
print(data_slice[data_slice.data_type=='sd'].value)

#### Quickly skim through several studies and check mean values:

In [ ]:
means = df[df.data_type== "mean"]
#means_author = means[means.publication== "Gratz & Roemer 2004"]
#means_author = means[means.publication== "Giesbrecht et al. 2007"]
#means_author = means[means.publication== "Levin & Spei 2003"]
#means_author = means[means.publication== "Giromini et al. 2012"]
#means_author = means[means.publication== "Neumann et al. 2010"]
#means_author = means[means.publication== "Hagborg et al. 2022"]
#means_author = means[means.publication== "Spitzer et al. 2014"]
means_author = means[means.publication== "Woodward et al. 2014"]
sample = means_author[means_author['sample_type']== 'patients']
sample_subsample = sample[sample['subsample']=='Site 2 (Perth)']
#sample_subsample = sample[sample['subsample']=='none']
sample_subsample_scale = sample_subsample[sample_subsample.scale_old=='PSYRATS-AHS_total']
sample_subsample_scale.value

In [ ]:
means_author

#### Median values:

In [ ]:
df[df.publication=="Haddock et al. 1999"].head()

In [ ]:
medians = df[df.data_type== "median"]

medians_author = medians[medians.publication=="Haddock et al. 1999"]
sample = medians_author[medians_author['sample_type']=='patients']
#sample_subsample = sample[sample['subsample']=='female']
#sample_subsample = sample[sample['subsample']=='none']
#sample_subsample_scale = sample_subsample[sample_subsample.scale=='PSYRATS-AS_total']
sample_subsample_scale = sample_subsample[sample_subsample.scale=='PSYRATS-AS_disruption']
sample_subsample_scale.value

#### Standard deviations:

In [ ]:
standard_deviations = df[df.data_type== "sd"]
#standard_deviations_author = standard_deviations[standard_deviations.publication== "Giesbrecht et al. 2007"]
#standard_deviations_author = standard_deviations[standard_deviations.publication== "Levin & Spei 2003"]
standard_deviations_author = standard_deviations[standard_deviations.publication== "Hagborg et al. 2022"]
sample = standard_deviations_author[standard_deviations_author['sample_type']== 'patients']
sample_subsample = sample[sample['subsample']=='clinical_girls']
#sample_subsample = sample[sample['subsample']=='none']
sample_subsample_scale = sample_subsample[sample_subsample.scale=='CTQ_PA']
sample_subsample_scale.value

In [ ]:
standard_deviations_author